# 02 - Global ReID and Camera Zones

This is the only Notebook 02. It reads the fresh four-camera run from Notebook 01, creates estimated global identities from appearance plus equal frame-time exports, then updates zone events and global annotated videos.

Zone mapping continues to use the existing camera calibration. Cross-camera identity does not ask for manual points: it uses appearance plus the shared FPS/frame-count timebase and labels every cross-camera result as an estimate.


In [1]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import json
import re

import cv2
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'Notebook':
    PROJECT_ROOT = PROJECT_ROOT.parent

TABLES_DIR = PROJECT_ROOT / 'Output' / 'tables'
VIDEOS_DIR = PROJECT_ROOT / 'Output' / 'videos'
CONFIG_DIR = PROJECT_ROOT / 'Data' / 'config'
LOCAL_TRACKS_PATH = TABLES_DIR / 'local_tracks.csv'
VIDEO_METADATA_PATH = TABLES_DIR / 'video_metadata.csv'
TRACKING_MANIFEST_PATH = TABLES_DIR / 'local_tracking_run.json'
CALIBRATION_PATH = CONFIG_DIR / 'camera_calibration.generated.json'
ZONES_PATH = CONFIG_DIR / 'store_zones.json'
REID_CONFIG_PATH = CONFIG_DIR / 'mtmc_reid_config.json'
REID_MANIFEST_PATH = TABLES_DIR / 'reid_run_manifest.json'

IDENTITY_CALIBRATION_LIMITS = {
    'min_points': 6,
    'min_inliers': 5,
    'min_inlier_ratio': 0.80,
    'max_mean_all_reprojection_error_px': 25.0,
}

def read_json(path):
    if not path.exists():
        raise FileNotFoundError(f'Missing required file: {path.relative_to(PROJECT_ROOT)}')
    return json.loads(path.read_text(encoding='utf-8'))


def write_json(path, payload):
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding='utf-8')


def infer_store_id(camera_id):
    match = re.search(r'(place_\d+)', str(camera_id), flags=re.IGNORECASE)
    return match.group(1).lower() if match else 'default_store'


def as_bool(value):
    if isinstance(value, str):
        return value.strip().casefold() in {'1', 'true', 'yes', 'y'}
    return bool(value)


def bool_series(values):
    return values.fillna(False).map(as_bool)


def validate_fresh_tracking():
    tracks = pd.read_csv(LOCAL_TRACKS_PATH)
    metadata = pd.read_csv(VIDEO_METADATA_PATH)
    manifest = read_json(TRACKING_MANIFEST_PATH)
    required = {
        'tracking_run_id', 'camera_id', 'frame_index', 'timestamp_sec',
        'local_track_id', 'confidence', 'x1', 'y1', 'x2', 'y2', 'foot_x', 'foot_y',
    }
    missing = sorted(required.difference(tracks.columns))
    if missing:
        raise RuntimeError(f'local_tracks.csv is not a fresh Notebook 01 output; missing {missing}. Run Notebook 01 again.')
    if 'tracking_run_id' not in metadata.columns:
        raise RuntimeError('video_metadata.csv is from an old run. Run Notebook 01 again before ReID.')
    run_ids = tracks['tracking_run_id'].dropna().astype(str).unique().tolist()
    metadata_run_ids = metadata['tracking_run_id'].dropna().astype(str).unique().tolist()
    if len(run_ids) != 1 or len(metadata_run_ids) != 1 or run_ids != metadata_run_ids:
        raise RuntimeError('Tracking tables contain mixed runs. Run Notebook 01 again; do not append cameras from another script.')
    run_id = run_ids[0]
    if str(manifest.get('tracking_run_id', '')) != run_id:
        raise RuntimeError('local_tracking_run.json does not belong to local_tracks.csv. Run Notebook 01 again.')
    tracks = tracks.copy()
    metadata = metadata.copy()
    tracks['camera_id'] = tracks['camera_id'].astype(str)
    metadata['camera_id'] = metadata['camera_id'].astype(str)
    tracks['local_track_id'] = pd.to_numeric(tracks['local_track_id'], errors='raise').astype(int)
    tracks['frame_index'] = pd.to_numeric(tracks['frame_index'], errors='raise').astype(int)
    tracks['timestamp_sec'] = pd.to_numeric(tracks['timestamp_sec'], errors='raise')
    camera_ids = set(tracks['camera_id'].unique())
    if camera_ids != set(metadata['camera_id'].unique()) or camera_ids != set(manifest.get('camera_ids', [])):
        raise RuntimeError('Camera lists disagree between local tracks, metadata, and manifest. Run Notebook 01 again.')
    camera_inputs = manifest.get('camera_inputs', {})
    if set(camera_inputs) != camera_ids:
        raise RuntimeError('The tracking manifest has no exact source-video list for this run. Run Notebook 01 again.')
    for camera_id, camera_rows in tracks.groupby('camera_id', sort=True):
        meta_rows = metadata.loc[metadata['camera_id'] == camera_id]
        if len(meta_rows) != 1:
            raise RuntimeError(f'Expected one metadata row for {camera_id}, found {len(meta_rows)}.')
        fps = float(meta_rows.iloc[0]['fps'])
        if not np.isfinite(fps) or fps <= 0:
            raise RuntimeError(f'Invalid FPS for {camera_id}.')
        expected = camera_rows['frame_index'].to_numpy(dtype=float) / fps
        observed = camera_rows['timestamp_sec'].to_numpy(dtype=float)
        max_error = float(np.max(np.abs(expected - observed)))
        if max_error > max(0.01, 1.5 / fps):
            raise RuntimeError(f'Wrong timestamp scale for {camera_id} (max error {max_error:.3f}s). Run Notebook 01 again.')
    return tracks, metadata, manifest, run_id


def validate_homography(entry, camera_id):
    matrix = np.asarray(entry.get('homography'), dtype=np.float64)
    if matrix.shape != (3, 3) or not np.isfinite(matrix).all() or np.linalg.matrix_rank(matrix) < 3:
        raise RuntimeError(f'Invalid homography for {camera_id}.')
    is_reference = as_bool(entry.get('reference_camera', False))
    if np.allclose(matrix, np.eye(3), atol=1e-9) and not is_reference:
        raise RuntimeError(f'Identity homography is allowed only for the declared reference camera: {camera_id}.')
    return matrix


def calibration_review(entry, camera_id):
    matrix = validate_homography(entry, camera_id)
    validation = entry.get('validation', {})
    is_reference = as_bool(entry.get('reference_camera', False))
    reasons = []
    if is_reference:
        if validation.get('status') != 'valid_reference_identity':
            reasons.append('reference camera is not explicitly validated')
    else:
        if validation.get('status') != 'valid':
            reasons.append('validation status is not valid')
        if int(validation.get('point_count', 0)) < IDENTITY_CALIBRATION_LIMITS['min_points']:
            reasons.append('not enough manual ground points')
        if int(validation.get('inlier_count', 0)) < IDENTITY_CALIBRATION_LIMITS['min_inliers']:
            reasons.append('not enough RANSAC inliers')
        if float(validation.get('inlier_ratio', 0.0)) < IDENTITY_CALIBRATION_LIMITS['min_inlier_ratio']:
            reasons.append('inlier ratio is too low')
        if float(validation.get('mean_all_reprojection_error_px', np.inf)) > IDENTITY_CALIBRATION_LIMITS['max_mean_all_reprojection_error_px']:
            reasons.append('manual ground points disagree')
    return {
        'camera_id': camera_id,
        'identity_ready': not reasons,
        'reference_camera': is_reference,
        'point_count': validation.get('point_count'),
        'inlier_count': validation.get('inlier_count'),
        'inlier_ratio': validation.get('inlier_ratio'),
        'condition_number': validation.get('condition_number'),
        'mean_all_reprojection_error_px': validation.get('mean_all_reprojection_error_px'),
        'reason': '; '.join(reasons),
        'homography': matrix,
    }


def assign_zone(floor_x, floor_y, zones):
    for zone in zones:
        polygon = np.asarray(zone['polygon'], dtype=np.float32)
        if cv2.pointPolygonTest(polygon, (float(floor_x), float(floor_y)), False) >= 0:
            return zone['zone_id'], zone['label_ar'], zone['kind']
    return 'outside', 'outside', 'outside'


LOCAL_TRACKS, VIDEO_METADATA, TRACKING_MANIFEST, TRACKING_RUN_ID = validate_fresh_tracking()
REID_CONFIG = read_json(REID_CONFIG_PATH)
IDENTITY_MODE = str(REID_CONFIG.get('association', {}).get('identity_mode', 'appearance_time')).strip().lower()
if IDENTITY_MODE not in {'appearance_time', 'calibrated_geometry'}:
    raise ValueError(f'Unsupported association.identity_mode: {IDENTITY_MODE}')
USE_GEOMETRY_FOR_IDENTITY = IDENTITY_MODE == 'calibrated_geometry'
CALIBRATIONS = read_json(CALIBRATION_PATH)
ZONES = read_json(ZONES_PATH)
if not isinstance(ZONES, list) or not ZONES:
    raise RuntimeError('store_zones.json must contain at least one zone.')

calibration_rows = []
calibration_matrices = {}
for camera_id in sorted(LOCAL_TRACKS['camera_id'].unique()):
    if camera_id not in CALIBRATIONS:
        raise RuntimeError(f'Missing calibration for {camera_id}.')
    review = calibration_review(CALIBRATIONS[camera_id], camera_id)
    calibration_matrices[camera_id] = review.pop('homography')
    calibration_rows.append(review)
CALIBRATION_DIAGNOSTICS = pd.DataFrame(calibration_rows)
CALIBRATION_DIAGNOSTICS.to_csv(TABLES_DIR / 'reid_calibration_diagnostics.csv', index=False)

BASE_EVENTS = LOCAL_TRACKS.copy()
BASE_EVENTS['store_id'] = BASE_EVENTS['camera_id'].map(infer_store_id)
BASE_EVENTS['is_employee'] = bool_series(BASE_EVENTS['is_employee']) if 'is_employee' in BASE_EVENTS else False
BASE_EVENTS['is_customer'] = ~BASE_EVENTS['is_employee']
BASE_EVENTS['camera_track_uid'] = (
    BASE_EVENTS['store_id'].astype(str) + '::' + BASE_EVENTS['camera_id'].astype(str) + '::' + BASE_EVENTS['local_track_id'].astype(str)
)
BASE_EVENTS['association_timestamp_sec'] = BASE_EVENTS['timestamp_sec'].astype(float)
for camera_id, offset in REID_CONFIG.get('synchronization', {}).get('camera_time_offsets_sec', {}).items():
    BASE_EVENTS.loc[BASE_EVENTS['camera_id'] == str(camera_id), 'association_timestamp_sec'] += float(offset)

for camera_id, camera_rows in BASE_EVENTS.groupby('camera_id', sort=True):
    points = camera_rows[['foot_x', 'foot_y']].to_numpy(dtype=np.float32).reshape(-1, 1, 2)
    mapped = cv2.perspectiveTransform(points, calibration_matrices[camera_id]).reshape(-1, 2)
    BASE_EVENTS.loc[camera_rows.index, 'floor_x'] = mapped[:, 0]
    BASE_EVENTS.loc[camera_rows.index, 'floor_y'] = mapped[:, 1]
BASE_EVENTS['calibration_mode'] = BASE_EVENTS['camera_id'].map(
    lambda camera_id: 'reference_camera' if as_bool(CALIBRATIONS[camera_id].get('reference_camera', False)) else 'configured'
)
zone_values = [assign_zone(x, y, ZONES) for x, y in BASE_EVENTS[['floor_x', 'floor_y']].to_numpy()]
BASE_EVENTS[['zone_id', 'zone_label_ar', 'zone_kind']] = pd.DataFrame(zone_values, index=BASE_EVENTS.index)

zone_columns = [
    'tracking_run_id', 'store_id', 'camera_id', 'camera_track_uid', 'local_track_id', 'is_employee', 'is_customer',
    'confidence', 'frame_index', 'timestamp_sec', 'foot_x', 'foot_y', 'x1', 'y1', 'x2', 'y2',
    'floor_x', 'floor_y', 'calibration_mode', 'zone_id', 'zone_label_ar', 'zone_kind',
]
BASE_ZONE_EVENTS = BASE_EVENTS[zone_columns].sort_values(['store_id', 'camera_id', 'camera_track_uid', 'frame_index']).reset_index(drop=True)
BASE_ZONE_EVENTS.to_csv(TABLES_DIR / 'zone_events.csv', index=False)
write_json(TABLES_DIR / 'camera_zone_run.json', {
    'schema_version': 2,
    'tracking_run_id': TRACKING_RUN_ID,
    'identity_scope': 'camera_local',
    'identity_note': 'camera_track_uid is the only analytics key until this notebook completes ReID',
    'camera_ids': sorted(BASE_EVENTS['camera_id'].unique().tolist()),
    'zone_event_rows': int(len(BASE_ZONE_EVENTS)),
})
write_json(REID_MANIFEST_PATH, {
    'schema_version': 2,
    'status': 'pending_calibration_review' if USE_GEOMETRY_FOR_IDENTITY else 'pending_appearance_time_reid',
    'tracking_run_id': TRACKING_RUN_ID,
    'camera_ids': sorted(BASE_EVENTS['camera_id'].unique().tolist()),
    'video_outputs': [],
})
bad_calibrations = CALIBRATION_DIAGNOSTICS.loc[~CALIBRATION_DIAGNOSTICS['identity_ready'], ['camera_id', 'reason']]
print(f'Camera-local zone events saved: {len(BASE_ZONE_EVENTS):,}')
if USE_GEOMETRY_FOR_IDENTITY:
    if bad_calibrations.empty:
        print('Calibration review passed. Run the ReID cell below.')
    else:
        print('Geometry ReID is blocked until these calibrations are corrected:')
        display(bad_calibrations)
else:
    print('Automatic Appearance + Time ReID is active. Zone calibration is retained only for zone assignment.')


Camera-local zone events saved: 93,002
ReID is blocked until these manual calibrations are corrected:


,camera_id,reason
1,CAFE_place_05_camera_18_15min,not enough RANSAC inliers; inlier ratio is too...
2,CAFE_place_05_camera_19_15min,not enough RANSAC inliers; inlier ratio is too...
3,CAFE_place_05_camera_20_15min,not enough RANSAC inliers; inlier ratio is too...


## Manual geometry calibration (optional legacy mode)

Appearance + Time is the default and does not use this picker. Leave it disabled. This cell remains only if you deliberately change association.identity_mode to calibrated_geometry later.

Use floor-tile corners, wall-to-floor corners, or furniture feet touching the floor only when you intentionally return to geometry mode.


In [ ]:
# Automatic Appearance + Time is the default. Keep this False so no click windows open.
RUN_MANUAL_CALIBRATION = False
CALIBRATION_FRAME_INDEX = 1500
POINTS_PER_CAMERA = 6
RESET_MANUAL_CALIBRATION_DRAFT = False  # Set True once only if you want to discard accepted cameras in this kernel.
PICKER_PANE_WIDTH = 800
PICKER_PANE_HEIGHT = 450
# The clicks are made on a resized view, so convert this visible tolerance back to source-image pixels.
RANSAC_DISPLAY_TOLERANCE_PX = 5.0
MAX_RANSAC_REPROJECTION_THRESHOLD_PX = 15.0
# Each target is clicked against a camera that has a real floor overlap with it.
# Cam20 is composed as Cam20 -> Cam19 -> Cam17, so it is never forced to match Cam17 directly.
CALIBRATION_PARENT_BY_CAMERA = {
    'CAFE_place_05_camera_18_15min': 'CAFE_place_05_camera_17_15min',
    'CAFE_place_05_camera_19_15min': 'CAFE_place_05_camera_17_15min',
    'CAFE_place_05_camera_20_15min': 'CAFE_place_05_camera_19_15min',
}

if RUN_MANUAL_CALIBRATION:
    required_names = ['PROJECT_ROOT', 'TRACKING_MANIFEST', 'VIDEO_METADATA', 'CALIBRATION_PATH']
    missing_names = [name for name in required_names if name not in globals()]
    if missing_names:
        raise RuntimeError('Run the calibration-review cell above before opening the point picker.')

    picker_limits = {
        'min_points': 6,
        'min_inliers': 5,
        'min_inlier_ratio': 0.80,
        'max_mean_all_reprojection_error_px': 25.0,
    }

    current_calibrations = read_json(CALIBRATION_PATH) if CALIBRATION_PATH.exists() else {}
    reference_ids = [
        str(camera_id) for camera_id, entry in current_calibrations.items()
        if as_bool(entry.get('reference_camera', False))
    ]
    if len(reference_ids) != 1:
        raise RuntimeError('The existing calibration must declare exactly one reference camera before point picking.')
    reference_camera_id = reference_ids[0]
    camera_ids = [str(camera_id) for camera_id in TRACKING_MANIFEST.get('camera_ids', [])]
    if reference_camera_id not in camera_ids:
        raise RuntimeError('The reference camera is not part of the fresh Notebook 01 tracking run.')
    target_camera_ids = [camera_id for camera_id in camera_ids if camera_id != reference_camera_id]
    if not target_camera_ids:
        raise RuntimeError('At least one non-reference camera is required for manual calibration.')
    if set(CALIBRATION_PARENT_BY_CAMERA) != set(target_camera_ids):
        raise RuntimeError('CALIBRATION_PARENT_BY_CAMERA must contain exactly one parent for every non-reference camera.')
    if any(parent_camera_id not in camera_ids for parent_camera_id in CALIBRATION_PARENT_BY_CAMERA.values()):
        raise RuntimeError('A calibration parent camera is missing from the fresh Notebook 01 run.')
    calibration_order, resolved_cameras = [], {reference_camera_id}
    unresolved_cameras = set(target_camera_ids)
    while unresolved_cameras:
        available = [
            camera_id for camera_id in target_camera_ids
            if camera_id in unresolved_cameras and CALIBRATION_PARENT_BY_CAMERA[camera_id] in resolved_cameras
        ]
        if not available:
            raise RuntimeError('Calibration parent graph has a cycle or no path to the reference camera.')
        calibration_order.extend(available)
        resolved_cameras.update(available)
        unresolved_cameras.difference_update(available)

    def picker_label(camera_id):
        token = str(camera_id).split('_camera_')[-1].split('_')[0]
        return f'Cam {token}'


    def resolve_picker_video(camera_id):
        relative_path = TRACKING_MANIFEST.get('camera_inputs', {}).get(camera_id)
        if not relative_path:
            raise RuntimeError(f'No source video is recorded for {camera_id}. Run Notebook 01 again.')
        path = (PROJECT_ROOT / Path(str(relative_path))).resolve()
        if not path.is_relative_to(PROJECT_ROOT) or not path.exists():
            raise RuntimeError(f'The recorded source video is unavailable for {camera_id}: {relative_path}')
        return path, str(relative_path).replace('\\', '/')


    def read_picker_frame(camera_id):
        path, relative_path = resolve_picker_video(camera_id)
        capture = cv2.VideoCapture(str(path))
        try:
            if not capture.isOpened():
                raise RuntimeError(f'Cannot open {path.name}.')
            total_frames = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
            if CALIBRATION_FRAME_INDEX < 0 or (total_frames > 0 and CALIBRATION_FRAME_INDEX >= total_frames):
                raise RuntimeError(f'Frame {CALIBRATION_FRAME_INDEX} is outside {path.name} ({total_frames} frames).')
            capture.set(cv2.CAP_PROP_POS_FRAMES, int(CALIBRATION_FRAME_INDEX))
            ok, frame = capture.read()
            if not ok or frame is None:
                raise RuntimeError(f'Cannot read frame {CALIBRATION_FRAME_INDEX} from {path.name}.')
            fps = float(capture.get(cv2.CAP_PROP_FPS))
        finally:
            capture.release()
        if not np.isfinite(fps) or fps <= 0:
            metadata_rows = VIDEO_METADATA.loc[VIDEO_METADATA['camera_id'].astype(str) == camera_id, 'fps']
            fps = float(metadata_rows.iloc[0]) if len(metadata_rows) == 1 else float('nan')
        if not np.isfinite(fps) or fps <= 0:
            raise RuntimeError(f'Invalid FPS for {camera_id}.')
        return frame, fps, relative_path


    def resize_for_picker(frame):
        height, width = frame.shape[:2]
        scale = min(1.0, PICKER_PANE_WIDTH / width, PICKER_PANE_HEIGHT / height)
        size = (max(1, round(width * scale)), max(1, round(height * scale)))
        view = cv2.resize(frame, size, interpolation=cv2.INTER_AREA) if scale < 1.0 else frame.copy()
        return view, scale


    def pick_six_pairs(pair_reference_frame, source_frame, pair_reference_camera_id, target_camera_id):
        reference_view, reference_scale = resize_for_picker(pair_reference_frame)
        source_view, source_scale = resize_for_picker(source_frame)
        header_height, gap = 76, 8
        reference_width, reference_height = reference_view.shape[1], reference_view.shape[0]
        source_width, source_height = source_view.shape[1], source_view.shape[0]
        source_x0 = reference_width + gap
        window_name = f'Calibration: {picker_label(pair_reference_camera_id)} <-> {picker_label(target_camera_id)}'
        colors = [(0, 255, 255), (0, 180, 0), (255, 180, 0), (255, 0, 255), (0, 130, 255), (255, 255, 0)]
        state = {'clicks': [], 'message': f'Click P1 in {picker_label(pair_reference_camera_id)} first.'}

        def point_from_canvas(side, x, y):
            local_y = y - header_height
            if side == 'reference':
                return float(x / reference_scale), float(local_y / reference_scale)
            return float((x - source_x0) / source_scale), float(local_y / source_scale)

        def side_at(x, y):
            local_y = y - header_height
            if 0 <= x < reference_width and 0 <= local_y < reference_height:
                return 'reference'
            if source_x0 <= x < source_x0 + source_width and 0 <= local_y < source_height:
                return 'source'
            return None

        def render():
            canvas = np.full((header_height + max(reference_height, source_height), source_x0 + source_width, 3), 24, dtype=np.uint8)
            canvas[header_height:header_height + reference_height, :reference_width] = reference_view
            canvas[header_height:header_height + source_height, source_x0:source_x0 + source_width] = source_view
            completed_pairs = len(state['clicks']) // 2
            if completed_pairs == POINTS_PER_CAMERA:
                line_one = f'{POINTS_PER_CAMERA}/{POINTS_PER_CAMERA} pairs complete. Press Enter to validate.  {state["message"]}'
            else:
                expected_side = picker_label(pair_reference_camera_id) if len(state['clicks']) % 2 == 0 else picker_label(target_camera_id)
                line_one = f'P{completed_pairs + 1}/{POINTS_PER_CAMERA}: click {expected_side}.  {state["message"]}'
            line_two = 'Right-click or U: undo | R: reset this camera | Enter: accept six pairs | Esc: cancel'
            cv2.putText(canvas, line_one, (14, 27), cv2.FONT_HERSHEY_SIMPLEX, 0.52, (255, 255, 255), 1, cv2.LINE_AA)
            cv2.putText(canvas, line_two, (14, 54), cv2.FONT_HERSHEY_SIMPLEX, 0.46, (190, 190, 190), 1, cv2.LINE_AA)
            cv2.putText(canvas, picker_label(pair_reference_camera_id), (12, header_height - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 255), 1, cv2.LINE_AA)
            cv2.putText(canvas, picker_label(target_camera_id), (source_x0 + 12, header_height - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 255), 1, cv2.LINE_AA)
            for click_index, (side, point) in enumerate(state['clicks']):
                point_number = click_index // 2 + 1
                color = colors[(point_number - 1) % len(colors)]
                if side == 'reference':
                    display_x = round(point[0] * reference_scale)
                    display_y = header_height + round(point[1] * reference_scale)
                else:
                    display_x = source_x0 + round(point[0] * source_scale)
                    display_y = header_height + round(point[1] * source_scale)
                cv2.circle(canvas, (display_x, display_y), 6, color, -1, cv2.LINE_AA)
                cv2.putText(canvas, f'P{point_number}', (display_x + 8, display_y - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.48, (0, 0, 0), 3, cv2.LINE_AA)
                cv2.putText(canvas, f'P{point_number}', (display_x + 8, display_y - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.48, color, 1, cv2.LINE_AA)
            return canvas

        def on_mouse(event, x, y, flags, param):
            if event == cv2.EVENT_RBUTTONDOWN:
                if state['clicks']:
                    state['clicks'].pop()
                    state['message'] = 'Last click removed.'
                return
            if event != cv2.EVENT_LBUTTONDOWN or len(state['clicks']) >= 2 * POINTS_PER_CAMERA:
                return
            clicked_side = side_at(x, y)
            expected_side = 'reference' if len(state['clicks']) % 2 == 0 else 'source'
            if clicked_side != expected_side:
                wanted = f'{picker_label(pair_reference_camera_id)} first' if expected_side == 'reference' else f'{picker_label(target_camera_id)} now'
                state['message'] = f'Wrong pane: click {wanted}.'
                return
            state['clicks'].append((clicked_side, point_from_canvas(clicked_side, x, y)))
            state['message'] = 'Pair recorded.'

        cv2.namedWindow(window_name, cv2.WINDOW_AUTOSIZE)
        cv2.setMouseCallback(window_name, on_mouse)
        try:
            while True:
                cv2.imshow(window_name, render())
                key = cv2.waitKey(20) & 0xFF
                if key == 27:
                    raise RuntimeError('Calibration cancelled; camera_calibration.generated.json was not changed.')
                if key in (ord('r'), ord('R')):
                    state['clicks'].clear()
                    state['message'] = 'This camera pair was reset.'
                elif key in (ord('u'), ord('U'), 8, 127) and state['clicks']:
                    state['clicks'].pop()
                    state['message'] = 'Last click removed.'
                elif key in (10, 13):
                    if len(state['clicks']) == 2 * POINTS_PER_CAMERA:
                        break
                    state['message'] = f'Choose all {POINTS_PER_CAMERA} pairs before pressing Enter.'
        finally:
            try:
                cv2.destroyWindow(window_name)
                cv2.waitKey(1)
            except cv2.error:
                pass
        reference_points = np.asarray([state['clicks'][index][1] for index in range(0, 2 * POINTS_PER_CAMERA, 2)], dtype=np.float32)
        source_points = np.asarray([state['clicks'][index][1] for index in range(1, 2 * POINTS_PER_CAMERA, 2)], dtype=np.float32)
        return source_points, reference_points, reference_scale


    def calibration_entry(camera_id, source_points, reference_points, source_fps, pair_reference_fps, source_video, pair_reference_camera_id, pair_reference_scale):
        if not np.isfinite(pair_reference_scale) or pair_reference_scale <= 0:
            return None, {'camera_id': camera_id, 'passed': False, 'reason': 'Invalid picker display scale.'}, None
        ransac_threshold_px = min(MAX_RANSAC_REPROJECTION_THRESHOLD_PX, RANSAC_DISPLAY_TOLERANCE_PX / float(pair_reference_scale))
        matrix, inliers = cv2.findHomography(source_points, reference_points, cv2.RANSAC, ransac_threshold_px)
        if matrix is None or inliers is None:
            return None, {'camera_id': camera_id, 'passed': False, 'reason': 'OpenCV could not estimate a homography.'}, None
        matrix = np.asarray(matrix, dtype=np.float64)
        inlier_mask = inliers.reshape(-1).astype(bool)
        projected = cv2.perspectiveTransform(source_points.reshape(-1, 1, 2), matrix).reshape(-1, 2)
        errors = np.linalg.norm(projected - reference_points, axis=1)
        rank = int(np.linalg.matrix_rank(matrix))
        condition_number = float(np.linalg.cond(matrix))
        inlier_count = int(inlier_mask.sum())
        inlier_ratio = float(inlier_mask.mean())
        mean_all_error = float(errors.mean())
        reasons = []
        if not np.isfinite(matrix).all() or rank < 3:
            reasons.append('invalid homography matrix')
        if inlier_count < picker_limits['min_inliers']:
            reasons.append(f'only {inlier_count}/{POINTS_PER_CAMERA} RANSAC inliers')
        if inlier_ratio < picker_limits['min_inlier_ratio']:
            reasons.append(f'inlier ratio {inlier_ratio:.2f} is below 0.80')
        if not np.isfinite(mean_all_error) or mean_all_error > picker_limits['max_mean_all_reprojection_error_px']:
            reasons.append(f'mean point error {mean_all_error:.1f}px is above 25px')
        passed = not reasons
        source_tag = picker_label(camera_id).replace(' ', '').lower()
        entry = {
            'homography': matrix.tolist(),
            'reference_camera': False,
            'reference_camera_id': pair_reference_camera_id,
            'edge_reference_camera_id': pair_reference_camera_id,
            'floor_units': 'reference_pixels',
            'mapping_direction': f'{camera_id}_pixels_to_{pair_reference_camera_id}_pixels',
            'calibration_method': 'manual_pair_homography_RANSAC',
            'source_video': source_video,
            'source_frame_index': int(CALIBRATION_FRAME_INDEX),
            'source_timestamp_sec': float(CALIBRATION_FRAME_INDEX / source_fps),
            'reference_frame_index': int(CALIBRATION_FRAME_INDEX),
            'reference_timestamp_sec': float(CALIBRATION_FRAME_INDEX / pair_reference_fps),
            'generated_at_utc': datetime.now(timezone.utc).isoformat(),
            'landmarks': [
                {
                    'name': f'{source_tag}_shared_ground_{index + 1:02d}',
                    'source_point': [float(value) for value in source_points[index]],
                    'reference_point': [float(value) for value in reference_points[index]],
                    'inlier': bool(inlier_mask[index]),
                    'reprojection_error_px': float(errors[index]),
                }
                for index in range(POINTS_PER_CAMERA)
            ],
            'validation': {
                'status': 'valid' if passed else 'invalid',
                'point_count': int(POINTS_PER_CAMERA),
                'inlier_count': inlier_count,
                'inlier_ratio': inlier_ratio,
                'mean_inlier_reprojection_error_px': float(errors[inlier_mask].mean()) if inlier_mask.any() else None,
                'median_inlier_reprojection_error_px': float(np.median(errors[inlier_mask])) if inlier_mask.any() else None,
                'max_inlier_reprojection_error_px': float(errors[inlier_mask].max()) if inlier_mask.any() else None,
                'mean_all_reprojection_error_px': mean_all_error,
                'condition_number': condition_number,
                'matrix_rank': rank,
                'determinant': float(np.linalg.det(matrix)),
                'validation_space': f'{pair_reference_camera_id}_pixels',
                'validation_errors': reasons,
                'thresholds': {
                    'ransac_reprojection_threshold_px': ransac_threshold_px,
                    'picker_display_tolerance_px': RANSAC_DISPLAY_TOLERANCE_PX,
                    'minimum_inlier_ratio': picker_limits['min_inlier_ratio'],
                    'maximum_mean_all_reprojection_error_px': picker_limits['max_mean_all_reprojection_error_px'],
                },
            },
        }
        summary = {
            'camera_id': camera_id,
            'passed': passed,
            'inliers': f'{inlier_count}/{POINTS_PER_CAMERA}',
            'condition_number': round(condition_number, 2),
            'mean_all_error_px': round(mean_all_error, 2),
            'reason': 'passed' if passed else '; '.join(reasons),
        }
        return entry, summary, projected


    def compose_entry_to_root(camera_id, edge_entry, pair_reference_camera_id, parent_entry):
        edge_matrix = np.asarray(edge_entry['homography'], dtype=np.float64)
        parent_matrix = np.eye(3, dtype=np.float64) if pair_reference_camera_id == reference_camera_id else np.asarray(parent_entry['homography'], dtype=np.float64)
        root_matrix = parent_matrix @ edge_matrix
        if not np.isfinite(root_matrix).all() or np.linalg.matrix_rank(root_matrix) < 3:
            return None
        entry = dict(edge_entry)
        entry['edge_homography'] = edge_matrix.tolist()
        entry['homography'] = root_matrix.tolist()
        entry['parent_camera_id'] = pair_reference_camera_id
        entry['reference_camera_id'] = reference_camera_id
        entry['mapping_direction'] = f'{camera_id}_pixels_to_{reference_camera_id}_pixels'
        entry['calibration_method'] = 'manual_camera_graph_composition_RANSAC'
        parent_path = [reference_camera_id] if pair_reference_camera_id == reference_camera_id else parent_entry.get('composition_path', [pair_reference_camera_id, reference_camera_id])
        entry['composition_path'] = [camera_id] + parent_path
        edge_reference_points = np.asarray([landmark['reference_point'] for landmark in entry['landmarks']], dtype=np.float32).reshape(-1, 1, 2)
        root_reference_points = cv2.perspectiveTransform(edge_reference_points, parent_matrix).reshape(-1, 2)
        for index, landmark in enumerate(entry['landmarks']):
            landmark['edge_reference_point'] = landmark['reference_point']
            landmark['reference_point'] = [float(value) for value in root_reference_points[index]]
        entry['validation'] = dict(edge_entry['validation'])
        entry['validation']['composed_condition_number'] = float(np.linalg.cond(root_matrix))
        entry['validation']['parent_camera_id'] = pair_reference_camera_id
        return entry


    def confirm_projection(pair_reference_frame, reference_points, projected_points, pair_reference_camera_id, target_camera_id):
        view, scale = resize_for_picker(pair_reference_frame)
        canvas = view.copy()
        for index, (reference_point, projected_point) in enumerate(zip(reference_points, projected_points), start=1):
            rx, ry = round(reference_point[0] * scale), round(reference_point[1] * scale)
            px, py = round(projected_point[0] * scale), round(projected_point[1] * scale)
            cv2.circle(canvas, (rx, ry), 6, (0, 255, 0), 2, cv2.LINE_AA)
            cv2.drawMarker(canvas, (px, py), (0, 0, 255), cv2.MARKER_TILTED_CROSS, 12, 2, cv2.LINE_AA)
            cv2.putText(canvas, f'P{index}', (rx + 8, ry - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.48, (255, 255, 255), 1, cv2.LINE_AA)
        window_name = f'Review {picker_label(target_camera_id)} -> {picker_label(pair_reference_camera_id)}'
        label = f'Green circle = {picker_label(pair_reference_camera_id)} click | red X = projected target point | Y: keep | R: pick again | Esc: cancel'
        cv2.putText(canvas, label, (12, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.40, (0, 0, 0), 3, cv2.LINE_AA)
        cv2.putText(canvas, label, (12, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.40, (255, 255, 255), 1, cv2.LINE_AA)
        cv2.namedWindow(window_name, cv2.WINDOW_AUTOSIZE)
        try:
            while True:
                cv2.imshow(window_name, canvas)
                key = cv2.waitKey(20) & 0xFF
                if key in (ord('y'), ord('Y')):
                    return True
                if key in (ord('r'), ord('R')):
                    return False
                if key == 27:
                    raise RuntimeError('Calibration cancelled; camera_calibration.generated.json was not changed.')
        finally:
            try:
                cv2.destroyWindow(window_name)
                cv2.waitKey(1)
            except cv2.error:
                pass


    draft_signature = {
        'tracking_run_id': str(TRACKING_MANIFEST.get('tracking_run_id', '')),
        'reference_camera_id': reference_camera_id,
        'target_camera_ids': tuple(calibration_order),
        'parent_edges': tuple((camera_id, CALIBRATION_PARENT_BY_CAMERA[camera_id]) for camera_id in calibration_order),
        'frame_index': int(CALIBRATION_FRAME_INDEX),
        'camera_inputs': tuple((camera_id, str(TRACKING_MANIFEST['camera_inputs'][camera_id])) for camera_id in camera_ids),
    }
    prior_draft = globals().get('MANUAL_CALIBRATION_DRAFT', {})
    if RESET_MANUAL_CALIBRATION_DRAFT or not isinstance(prior_draft, dict) or prior_draft.get('signature') != draft_signature:
        prior_draft = {'signature': draft_signature, 'accepted_entries': {}}
    accepted_entries = prior_draft.get('accepted_entries', {})
    selected_entries = {
        camera_id: entry for camera_id, entry in accepted_entries.items()
        if camera_id in calibration_order
    } if isinstance(accepted_entries, dict) else {}
    MANUAL_CALIBRATION_DRAFT = {'signature': draft_signature, 'accepted_entries': dict(selected_entries)}
    reference_frame, reference_fps, reference_video = read_picker_frame(reference_camera_id)
    summary_rows = []

    for target_camera_id in calibration_order:
        if target_camera_id in selected_entries:
            summary_rows.append({
                'camera_id': target_camera_id,
                'passed': True,
                'reason': 'already accepted in this notebook session',
            })
            print(f'{picker_label(target_camera_id)} already passed; moving to the next camera.')
            continue
        pair_reference_camera_id = CALIBRATION_PARENT_BY_CAMERA[target_camera_id]
        if pair_reference_camera_id == reference_camera_id:
            pair_reference_frame, pair_reference_fps = reference_frame, reference_fps
            parent_entry = None
        else:
            parent_entry = selected_entries.get(pair_reference_camera_id)
            if parent_entry is None:
                summary_rows.append({
                    'camera_id': target_camera_id,
                    'passed': False,
                    'reason': f'waiting for {picker_label(pair_reference_camera_id)} to pass first',
                })
                print(f'{picker_label(target_camera_id)} is skipped for now because {picker_label(pair_reference_camera_id)} has not passed.')
                continue
            pair_reference_frame, pair_reference_fps, _ = read_picker_frame(pair_reference_camera_id)
        source_frame, source_fps, source_video = read_picker_frame(target_camera_id)
        print(f'Choose six pairs for {picker_label(target_camera_id)}. Each pair is {picker_label(pair_reference_camera_id)} first, then {picker_label(target_camera_id)}.')
        source_points, reference_points, pair_reference_scale = pick_six_pairs(pair_reference_frame, source_frame, pair_reference_camera_id, target_camera_id)
        edge_entry, summary, projected_points = calibration_entry(
            target_camera_id, source_points, reference_points, source_fps, pair_reference_fps, source_video, pair_reference_camera_id, pair_reference_scale
        )
        entry = compose_entry_to_root(target_camera_id, edge_entry, pair_reference_camera_id, parent_entry) if edge_entry is not None and summary['passed'] else None
        if entry is None and summary['passed']:
            summary['passed'] = False
            summary['reason'] = 'could not compose this camera path to Cam17'
        if entry is not None and summary['passed']:
            if confirm_projection(pair_reference_frame, reference_points, projected_points, pair_reference_camera_id, target_camera_id):
                selected_entries[target_camera_id] = entry
                MANUAL_CALIBRATION_DRAFT = {'signature': draft_signature, 'accepted_entries': dict(selected_entries)}
                summary['reason'] = 'accepted'
            else:
                summary['passed'] = False
                summary['reason'] = 'review was rejected; select this camera again on the next run'
        summary_rows.append(summary)
        if not summary['passed']:
            print(f'{picker_label(target_camera_id)} did not pass. Continuing to the next camera; it will be the only one reopened when you run this cell again.')

    display(pd.DataFrame(summary_rows))
    missing_camera_ids = [camera_id for camera_id in calibration_order if camera_id not in selected_entries]
    if missing_camera_ids:
        MANUAL_CALIBRATION_DRAFT = {'signature': draft_signature, 'accepted_entries': dict(selected_entries)}
        pending_labels = ', '.join(picker_label(camera_id) for camera_id in missing_camera_ids)
        print(f'Nothing was saved yet. Run this same cell again to reselect only: {pending_labels}.')
    else:
        reference_entry = {
            'homography': np.eye(3, dtype=float).tolist(),
            'reference_camera': True,
            'reference_camera_id': reference_camera_id,
            'floor_units': 'reference_pixels',
            'mapping_direction': 'identity_reference_pixels',
            'calibration_method': 'explicit_reference_identity',
            'source_video': reference_video,
            'source_frame_index': int(CALIBRATION_FRAME_INDEX),
            'source_timestamp_sec': float(CALIBRATION_FRAME_INDEX / reference_fps),
            'generated_at_utc': datetime.now(timezone.utc).isoformat(),
            'validation': {
                'status': 'valid_reference_identity',
                'condition_number': 1.0,
                'matrix_rank': 3,
                'determinant': 1.0,
            },
        }
        payload = dict(current_calibrations)
        payload[reference_camera_id] = reference_entry
        payload.update(selected_entries)
        temporary_path = CALIBRATION_PATH.with_suffix('.tmp')
        temporary_path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding='utf-8')
        temporary_path.replace(CALIBRATION_PATH)
        reid_config = read_json(REID_CONFIG_PATH)
        reid_config.setdefault('association', {})['allowed_camera_pairs'] = [
            [camera_id, CALIBRATION_PARENT_BY_CAMERA[camera_id]] for camera_id in calibration_order
        ]
        write_json(REID_CONFIG_PATH, reid_config)
        MANUAL_CALIBRATION_DRAFT = {}
        print(f'Saved reviewed six-point calibrations for {len(selected_entries)} cameras to {CALIBRATION_PATH.relative_to(PROJECT_ROOT)}.')
        print('ReID is restricted to the reviewed overlapping camera pairs from this calibration graph.')
        print('Now rerun the calibration-review cell above. Video synchronization remains deliberately unverified.')
else:
    print('Manual point picker is disabled. Appearance + Time is the default mode.')


Manual point picker is disabled. Appearance + Time is the default mode.


## Re-identification

This cell has no embedding cache: every successful run is tied to the exact tracking_run_id. It uses synchronized ground-plane distance for overlapping tracks, endpoint distance only for real handoffs, Hungarian matching per camera pair, and refuses any cluster that contains overlapping local tracks from the same camera.


In [3]:
import importlib.util
import math
import sys
from itertools import combinations

import torch
from torch.nn import functional as torch_functional

NOTEBOOK_DIR = PROJECT_ROOT / 'Notebook'
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))
from reid_balanced import (
    aggregate_local_components, annotate_mapping_quality, build_local_components, color_for_identity,
    interpolate_display_rows, select_disjoint_local_stitches,
    select_mutual_best_matches, sha256_file, synchronized_cosine_distance,
)

if USE_GEOMETRY_FOR_IDENTITY and not bool(CALIBRATION_DIAGNOSTICS['identity_ready'].all()):
    write_json(REID_MANIFEST_PATH, {
        'schema_version': 2,
        'status': 'blocked_calibration',
        'tracking_run_id': TRACKING_RUN_ID,
        'camera_ids': sorted(BASE_EVENTS['camera_id'].unique().tolist()),
        'video_outputs': [],
        'diagnostics': 'Output/tables/reid_calibration_diagnostics.csv',
    })
    raise RuntimeError('ReID stopped: correct the cameras marked false in reid_calibration_diagnostics.csv, then rerun this cell.')

if USE_GEOMETRY_FOR_IDENTITY and not as_bool(REID_CONFIG.get('synchronization', {}).get('verified', False)):
    write_json(REID_MANIFEST_PATH, {
        'schema_version': 2,
        'status': 'blocked_synchronization',
        'tracking_run_id': TRACKING_RUN_ID,
        'camera_ids': sorted(BASE_EVENTS['camera_id'].unique().tolist()),
        'video_outputs': [],
        'diagnostics': 'Set synchronization.verified only after checking the shared video start and offsets.',
    })
    raise RuntimeError('ReID stopped: verify that all camera clips share the same time zero, then set synchronization.verified in mtmc_reid_config.json.')

def verify_appearance_timebase(metadata, camera_ids):
    columns = ['camera_id', 'fps', 'total_frames', 'duration_sec']
    rows = metadata.loc[metadata['camera_id'].astype(str).isin(camera_ids), columns].copy()
    if rows['camera_id'].astype(str).nunique() != len(camera_ids):
        raise RuntimeError('Appearance + Time requires metadata for every tracked camera.')
    fps_values = pd.to_numeric(rows['fps'], errors='coerce')
    frame_counts = pd.to_numeric(rows['total_frames'], errors='coerce')
    durations = pd.to_numeric(rows['duration_sec'], errors='coerce')
    if not (np.isfinite(fps_values).all() and np.isfinite(frame_counts).all() and np.isfinite(durations).all()):
        raise RuntimeError('Appearance + Time needs finite FPS, frame count, and duration metadata.')
    if fps_values.nunique() != 1 or frame_counts.nunique() != 1 or durations.nunique() != 1:
        raise RuntimeError('Appearance + Time requires the selected camera exports to share FPS, frame count, and duration.')
    return {
        'mode': 'equal_frame_grid_assumed',
        'fps': float(fps_values.iloc[0]),
        'total_frames': int(frame_counts.iloc[0]),
        'duration_sec': float(durations.iloc[0]),
        'camera_ids': sorted(camera_ids),
    }

if IDENTITY_MODE == 'appearance_time':
    TIMEBASE_REPORT = verify_appearance_timebase(VIDEO_METADATA, sorted(BASE_EVENTS['camera_id'].astype(str).unique().tolist()))
    print('Appearance + Time uses the shared FPS/frame-count grid; cross-camera IDs are estimates.')
else:
    TIMEBASE_REPORT = {'mode': 'verified_external_offsets', 'camera_time_offsets_sec': REID_CONFIG.get('synchronization', {}).get('camera_time_offsets_sec', {})}

REID_SETTINGS = REID_CONFIG.get('reid', {})
ASSOCIATION = REID_CONFIG.get('association', {})

def make_tracklet_id(store_id, camera_id, local_track_id):
    return f'{store_id}::{camera_id}::{int(local_track_id)}'


def resolve_input_videos(camera_ids):
    declared = TRACKING_MANIFEST.get('camera_inputs', {})
    resolved = {}
    for camera_id in sorted(map(str, camera_ids)):
        relative_path = declared.get(camera_id)
        if relative_path is None:
            raise RuntimeError(f'No source video recorded for {camera_id}.')
        video_path = (PROJECT_ROOT / Path(str(relative_path))).resolve()
        if not video_path.is_relative_to(PROJECT_ROOT) or not video_path.exists():
            raise RuntimeError(f'Recorded source video is unavailable for {camera_id}: {relative_path}')
        resolved[camera_id] = video_path
    return resolved


def load_osnet():
    source_path = PROJECT_ROOT / 'vendor' / 'torchreid_osnet' / 'osnet_ain.py'
    weights_path = PROJECT_ROOT / Path(str(REID_SETTINGS['weights_path']))
    if not source_path.exists() or not weights_path.exists():
        raise FileNotFoundError('OSNet source or local weights are missing; no model will be downloaded automatically.')
    spec = importlib.util.spec_from_file_location('retail_osnet_ain', source_path)
    if spec is None or spec.loader is None:
        raise RuntimeError('Could not load the vendored OSNet source.')
    module = importlib.util.module_from_spec(spec)
    sys.modules[spec.name] = module
    spec.loader.exec_module(module)
    requested_device = str(REID_SETTINGS.get('device', 'auto'))
    device = torch.device('cuda' if requested_device == 'auto' and torch.cuda.is_available() else requested_device if requested_device != 'auto' else 'cpu')
    model = module.osnet_ain_x1_0(num_classes=1000, pretrained=False)
    try:
        checkpoint = torch.load(weights_path, map_location='cpu', weights_only=True)
    except TypeError:
        checkpoint = torch.load(weights_path, map_location='cpu')
    state = checkpoint.get('state_dict', checkpoint) if isinstance(checkpoint, dict) else checkpoint
    model_state = model.state_dict()
    compatible = {
        str(key).removeprefix('module.'): value
        for key, value in state.items()
        if str(key).removeprefix('module.') in model_state
        and model_state[str(key).removeprefix('module.')].shape == value.shape
    }
    if not compatible:
        raise RuntimeError('The local OSNet checkpoint does not match the vendored model.')
    model.load_state_dict(compatible, strict=False)
    return model.to(device).eval(), device


def pick_embedding_samples(events):
    rows = events.copy()
    rows['box_area'] = (rows['x2'] - rows['x1']) * (rows['y2'] - rows['y1'])
    rows = rows.loc[
        (rows['confidence'] >= float(REID_SETTINGS.get('min_confidence', 0.25)))
        & (rows['box_area'] >= float(REID_SETTINGS.get('min_box_area', 1024.0)))
    ].copy()
    rows['tracklet_id'] = [
        make_tracklet_id(store_id, camera_id, local_track_id)
        for store_id, camera_id, local_track_id in rows[['store_id', 'camera_id', 'local_track_id']].itertuples(index=False, name=None)
    ]
    chosen = []
    maximum = int(REID_SETTINGS.get('samples_per_tracklet', 12))
    minimum_separation = float(REID_SETTINGS.get('min_time_separation_sec', 0.5))
    for _, group in rows.sort_values(['tracklet_id', 'association_timestamp_sec', 'frame_index']).groupby('tracklet_id', sort=True):
        positions = np.unique(np.linspace(0, len(group) - 1, min(maximum, len(group)), dtype=int))
        selected = []
        for position in positions:
            candidate = group.iloc[int(position)]
            if not selected or float(candidate['association_timestamp_sec']) - float(selected[-1]['association_timestamp_sec']) >= minimum_separation:
                selected.append(candidate)
        if not selected:
            selected.append(group.iloc[len(group) // 2])
        chosen.extend(selected)
    return pd.DataFrame(chosen)


def extract_embeddings(events, model, device):
    samples = pick_embedding_samples(events)
    if samples.empty:
        return {}, {}, {'requested_crops': 0, 'valid_crops': 0, 'embedded_tracklets': 0}
    videos = resolve_input_videos(samples['camera_id'].unique())
    crop_margin = float(REID_SETTINGS.get('crop_margin', 0.05))
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    crops = {}
    valid_crops = 0
    for camera_id, camera_rows in samples.groupby('camera_id', sort=True):
        targets = {}
        for row in camera_rows.to_dict('records'):
            targets.setdefault(int(row['frame_index']), []).append(row)
        capture = cv2.VideoCapture(str(videos[str(camera_id)]))
        if not capture.isOpened():
            raise RuntimeError(f'Could not open the recorded source video for {camera_id}.')
        try:
            for frame_index in range(max(targets) + 1):
                ok, frame = capture.read()
                if not ok or frame is None:
                    break
                for row in targets.get(frame_index, []):
                    height, width = frame.shape[:2]
                    x1, y1, x2, y2 = map(float, (row['x1'], row['y1'], row['x2'], row['y2']))
                    pad_x, pad_y = (x2 - x1) * crop_margin, (y2 - y1) * crop_margin
                    x1, y1 = max(0, int(math.floor(x1 - pad_x))), max(0, int(math.floor(y1 - pad_y)))
                    x2, y2 = min(width, int(math.ceil(x2 + pad_x))), min(height, int(math.ceil(y2 + pad_y)))
                    if x2 <= x1 or y2 <= y1:
                        continue
                    image = cv2.cvtColor(cv2.resize(frame[y1:y2, x1:x2], (128, 256)), cv2.COLOR_BGR2RGB)
                    tensor = torch.from_numpy(np.ascontiguousarray(image)).permute(2, 0, 1).float().div(255.0)
                    bucket = crops.setdefault(str(row['tracklet_id']), {'times': [], 'tensors': []})
                    bucket['times'].append(float(row['association_timestamp_sec']))
                    bucket['tensors'].append((tensor - mean) / std)
                    valid_crops += 1
        finally:
            capture.release()
    embeddings = {}
    embedding_samples = {}
    batch_size = int(REID_SETTINGS.get('batch_size', 32))
    with torch.inference_mode():
        for tracklet_id, bucket in crops.items():
            tensors = bucket['tensors']
            vectors = []
            for start in range(0, len(tensors), batch_size):
                features = model(torch.stack(tensors[start:start + batch_size]).to(device))
                if isinstance(features, (list, tuple)):
                    features = features[0]
                vectors.append(torch_functional.normalize(features, p=2, dim=1).cpu())
            matrix = torch.cat(vectors)
            embeddings[tracklet_id] = torch_functional.normalize(matrix.mean(dim=0), p=2, dim=0).numpy()
            embedding_samples[tracklet_id] = {
                'times': np.asarray(bucket['times'], dtype=float),
                'vectors': matrix.numpy(),
            }
    return embeddings, embedding_samples, {
        'requested_crops': int(len(samples)),
        'valid_crops': int(valid_crops),
        'embedded_tracklets': int(len(embeddings)),
    }


def build_tracklets(events):
    summaries = []
    floor_points = {}
    pixel_points = {}
    for (store_id, camera_id, local_track_id), group in events.sort_values(['camera_id', 'local_track_id', 'association_timestamp_sec', 'frame_index']).groupby(['store_id', 'camera_id', 'local_track_id'], sort=True):
        group = group.reset_index(drop=True)
        tracklet_id = make_tracklet_id(store_id, camera_id, local_track_id)
        floor_timeline = group[['association_timestamp_sec', 'floor_x', 'floor_y']].dropna().rename(columns={'association_timestamp_sec': 'time_sec'})
        pixel_timeline = group[['association_timestamp_sec', 'foot_x', 'foot_y']].dropna().rename(columns={'association_timestamp_sec': 'time_sec', 'foot_x': 'pixel_x', 'foot_y': 'pixel_y'})
        if floor_timeline.empty or pixel_timeline.empty:
            continue
        floor_points[tracklet_id] = floor_timeline.sort_values('time_sec').reset_index(drop=True)
        pixel_points[tracklet_id] = pixel_timeline.sort_values('time_sec').reset_index(drop=True)
        summaries.append({
            'store_id': str(store_id),
            'camera_id': str(camera_id),
            'local_track_id': int(local_track_id),
            'tracklet_id': tracklet_id,
            'start_sec': float(pixel_timeline['time_sec'].iloc[0]),
            'end_sec': float(pixel_timeline['time_sec'].iloc[-1]),
            'is_employee': bool(group['is_employee'].mode(dropna=False).iloc[0]),
        })
    return pd.DataFrame(summaries), floor_points, pixel_points


def cosine_distance(left, right):
    return float(1.0 - np.clip(np.dot(left, right), -1.0, 1.0))


def aligned_spatial_distance(left_points, right_points):
    overlap_start = max(float(left_points['time_sec'].min()), float(right_points['time_sec'].min()))
    overlap_end = min(float(left_points['time_sec'].max()), float(right_points['time_sec'].max()))
    if overlap_start <= overlap_end:
        right = right_points.rename(columns={'time_sec': 'other_time', 'floor_x': 'other_x', 'floor_y': 'other_y'})
        aligned = pd.merge_asof(
            left_points.sort_values('time_sec'), right.sort_values('other_time'),
            left_on='time_sec', right_on='other_time', direction='nearest',
            tolerance=float(ASSOCIATION.get('synchronization_tolerance_sec', 0.35)),
        ).dropna(subset=['other_x', 'other_y'])
        minimum = int(ASSOCIATION.get('min_synchronized_samples', 3))
        if len(aligned) < minimum:
            return np.nan, int(len(aligned)), 'insufficient_synchronization', 0.0
        maximum = int(ASSOCIATION.get('max_synchronized_samples', 25))
        if len(aligned) > maximum:
            aligned = aligned.iloc[np.linspace(0, len(aligned) - 1, maximum, dtype=int)]
        distance = np.hypot(aligned['floor_x'] - aligned['other_x'], aligned['floor_y'] - aligned['other_y'])
        return float(np.median(distance)), int(len(aligned)), 'synchronized_median', 0.0
    if float(left_points['time_sec'].max()) < float(right_points['time_sec'].min()):
        first, second = left_points.iloc[-1], right_points.iloc[0]
    else:
        first, second = right_points.iloc[-1], left_points.iloc[0]
    distance = math.hypot(float(first['floor_x']) - float(second['floor_x']), float(first['floor_y']) - float(second['floor_y']))
    gap = abs(float(second['time_sec']) - float(first['time_sec']))
    return float(distance), 0, 'handoff_endpoint', gap


def local_pixel_handoff(left_points, right_points):
    if float(left_points['time_sec'].max()) < float(right_points['time_sec'].min()):
        first, second = left_points.iloc[-1], right_points.iloc[0]
    elif float(right_points['time_sec'].max()) < float(left_points['time_sec'].min()):
        first, second = right_points.iloc[-1], left_points.iloc[0]
    else:
        return float('nan'), 0.0
    distance = math.hypot(float(first['pixel_x']) - float(second['pixel_x']), float(first['pixel_y']) - float(second['pixel_y']))
    gap = abs(float(second['time_sec']) - float(first['time_sec']))
    return float(distance), float(gap)


CANDIDATE_COLUMNS = [
    'store_id', 'camera_id_a', 'camera_id_b', 'tracklet_id_a', 'tracklet_id_b', 'association_time_sec',
    'score', 'appearance_distance', 'floor_distance', 'pixel_distance', 'time_gap_sec', 'temporal_overlap_sec', 'synchronized_samples', 'spatial_method', 'identity_mode', 'is_cross_camera',
]


def build_candidates(tracklets, floor_points, pixel_points, embedding_samples, *, include_same_camera=True, include_cross_camera=True):
    allowed_raw = ASSOCIATION.get('allowed_camera_pairs')
    allowed_pairs = None if not allowed_raw else {frozenset(map(str, pair)) for pair in allowed_raw}
    records = tracklets.to_dict('records')
    candidates = []
    for left, right in combinations(records, 2):
        if left['store_id'] != right['store_id']:
            continue
        same_camera = left['camera_id'] == right['camera_id']
        if same_camera and not include_same_camera:
            continue
        if not same_camera and not include_cross_camera:
            continue
        if same_camera and not as_bool(ASSOCIATION.get('allow_same_camera_nonoverlap', True)):
            continue
        if not same_camera and allowed_pairs is not None and frozenset((left['camera_id'], right['camera_id'])) not in allowed_pairs:
            continue
        if as_bool(ASSOCIATION.get('require_same_employee_label', True)) and left['is_employee'] != right['is_employee']:
            continue
        max_gap = float(ASSOCIATION.get('same_camera_max_time_gap_sec', 10.0) if same_camera else ASSOCIATION.get('max_time_gap_sec', 12.0))
        max_floor = float(ASSOCIATION.get('max_floor_distance', 120.0))
        max_appearance = float(ASSOCIATION.get('same_camera_max_appearance_distance', 0.25) if same_camera else ASSOCIATION.get('max_appearance_distance', 0.45))
        coarse_time_gap = max(0.0, left['start_sec'] - right['end_sec'], right['start_sec'] - left['end_sec'])
        temporal_overlap_sec = max(0.0, min(left['end_sec'], right['end_sec']) - max(left['start_sec'], right['start_sec']))
        if coarse_time_gap > max_gap:
            continue
        mean_appearance_distance = cosine_distance(left['embedding'], right['embedding'])
        if same_camera:
            pixel_distance, time_gap = local_pixel_handoff(pixel_points[left['tracklet_id']], pixel_points[right['tracklet_id']])
            max_pixel_distance = float(ASSOCIATION.get('same_camera_max_pixel_distance', 180.0))
            if not np.isfinite(pixel_distance) or time_gap > max_gap or pixel_distance > max_pixel_distance or mean_appearance_distance > max_appearance:
                continue
            appearance_distance = mean_appearance_distance
            floor_distance = float('nan')
            synchronized_samples = 0
            spatial_method = 'camera_pixel_handoff'
            score = (
                float(ASSOCIATION.get('same_camera_appearance_weight', 0.75)) * appearance_distance / max_appearance
                + float(ASSOCIATION.get('same_camera_pixel_weight', 0.15)) * pixel_distance / max_pixel_distance
                + float(ASSOCIATION.get('same_camera_time_weight', 0.10)) * time_gap / max_gap
            )
        elif IDENTITY_MODE == 'appearance_time':
            if temporal_overlap_sec < float(ASSOCIATION.get('min_temporal_overlap_sec', 0.6)):
                continue
            left_samples = embedding_samples.get(left['tracklet_id'])
            right_samples = embedding_samples.get(right['tracklet_id'])
            if left_samples is None or right_samples is None:
                continue
            appearance_distance, synchronized_samples = synchronized_cosine_distance(
                left_samples['times'], left_samples['vectors'], right_samples['times'], right_samples['vectors'],
                tolerance_sec=float(ASSOCIATION.get('synchronization_tolerance_sec', 0.35)),
                min_samples=int(ASSOCIATION.get('min_synchronized_samples', 3)),
                max_samples=int(ASSOCIATION.get('max_synchronized_samples', 25)),
            )
            if not np.isfinite(appearance_distance) or appearance_distance > max_appearance:
                continue
            floor_distance = float('nan')
            pixel_distance = float('nan')
            spatial_method = 'synchronized_appearance_samples'
            time_gap = coarse_time_gap
            score = appearance_distance / max_appearance
        else:
            appearance_distance = mean_appearance_distance
            pixel_distance = float('nan')
            if appearance_distance > max_appearance:
                continue
            floor_distance, synchronized_samples, spatial_method, time_gap = aligned_spatial_distance(floor_points[left['tracklet_id']], floor_points[right['tracklet_id']])
            if not np.isfinite(floor_distance) or time_gap > max_gap or floor_distance > max_floor:
                continue
            score = (
                float(ASSOCIATION.get('appearance_weight', 0.65)) * appearance_distance / max_appearance
                + float(ASSOCIATION.get('spatial_weight', 0.25)) * floor_distance / max_floor
                + float(ASSOCIATION.get('temporal_weight', 0.10)) * time_gap / max_gap
            )
        if same_camera:
            first, second = (left, right) if (left['start_sec'], left['tracklet_id']) <= (right['start_sec'], right['tracklet_id']) else (right, left)
        else:
            first, second = (left, right) if (left['camera_id'], left['tracklet_id']) <= (right['camera_id'], right['tracklet_id']) else (right, left)
        candidates.append({
            'store_id': first['store_id'], 'camera_id_a': first['camera_id'], 'camera_id_b': second['camera_id'],
            'tracklet_id_a': first['tracklet_id'], 'tracklet_id_b': second['tracklet_id'],
            'association_time_sec': max(first['start_sec'], second['start_sec']), 'score': float(score),
            'appearance_distance': float(appearance_distance), 'floor_distance': float(floor_distance), 'pixel_distance': float(pixel_distance),
            'time_gap_sec': float(time_gap), 'temporal_overlap_sec': float(temporal_overlap_sec),
            'synchronized_samples': synchronized_samples, 'spatial_method': spatial_method,
            'identity_mode': IDENTITY_MODE, 'is_cross_camera': not same_camera,
        })
    return pd.DataFrame(candidates, columns=CANDIDATE_COLUMNS)


def cluster_matches(tracklets, matches):
    records = {row['tracklet_id']: row for row in tracklets.to_dict('records')}
    parent = {tracklet_id: tracklet_id for tracklet_id in records}
    def find(tracklet_id):
        while parent[tracklet_id] != tracklet_id:
            parent[tracklet_id] = parent[parent[tracklet_id]]
            tracklet_id = parent[tracklet_id]
        return tracklet_id
    def members(tracklet_id):
        root = find(tracklet_id)
        return [candidate for candidate in parent if find(candidate) == root]
    def conflict(left_members, right_members):
        for left_id, right_id in combinations(left_members + right_members, 2):
            left, right = records[left_id], records[right_id]
            if left['camera_id'] != right['camera_id']:
                continue
            overlap = not (
                left['end_sec'] + float(ASSOCIATION.get('overlap_tolerance_sec', 0.0)) < right['start_sec']
                or right['end_sec'] + float(ASSOCIATION.get('overlap_tolerance_sec', 0.0)) < left['start_sec']
            )
            if overlap:
                return True
        return False
    accepted, rejected = [], []
    for match in matches.to_dict('records'):
        left_id, right_id = match['tracklet_id_a'], match['tracklet_id_b']
        left_members, right_members = members(left_id), members(right_id)
        if set(left_members) == set(right_members):
            continue
        if conflict(left_members, right_members):
            rejected.append({**match, 'rejection_reason': 'overlapping_same_camera_tracklets'})
            continue
        parent[find(right_id)] = find(left_id)
        accepted.append(match)
    groups = {}
    for tracklet_id in parent:
        groups.setdefault(find(tracklet_id), []).append(tracklet_id)
    global_ids = {}
    group_camera_counts = {}
    for index, members_list in enumerate(sorted(groups.values(), key=lambda values: min(values)), start=1):
        global_id = f'global_{index:06d}'
        group_camera_counts[global_id] = len({records[tracklet_id]['camera_id'] for tracklet_id in members_list})
        for tracklet_id in members_list:
            global_ids[tracklet_id] = global_id
    mapping = tracklets[['store_id', 'camera_id', 'local_track_id', 'tracklet_id']].copy()
    mapping['global_track_id'] = mapping['tracklet_id'].map(global_ids)
    mapping['is_cross_camera_identity'] = mapping['global_track_id'].map(lambda global_id: group_camera_counts[global_id] > 1)
    return mapping, pd.DataFrame(accepted, columns=matches.columns), pd.DataFrame(rejected, columns=[*matches.columns, 'rejection_reason'])


TRACKLETS, TRACKLET_FLOOR_POINTS, TRACKLET_PIXEL_POINTS = build_tracklets(BASE_EVENTS)
if TRACKLETS.empty:
    raise RuntimeError('No usable local tracklets were produced.')
MODEL, DEVICE = load_osnet()
EMBEDDINGS, EMBEDDING_SAMPLES, EMBEDDING_REPORT = extract_embeddings(BASE_EVENTS, MODEL, DEVICE)
MATCHABLE_TRACKLETS = TRACKLETS.loc[TRACKLETS['tracklet_id'].isin(EMBEDDINGS)].copy()
MATCHABLE_TRACKLETS['embedding'] = MATCHABLE_TRACKLETS['tracklet_id'].map(EMBEDDINGS)
LOCAL_CANDIDATES = build_candidates(
    MATCHABLE_TRACKLETS, TRACKLET_FLOOR_POINTS, TRACKLET_PIXEL_POINTS, EMBEDDING_SAMPLES,
    include_same_camera=True, include_cross_camera=False,
)
LOCAL_MATCHES = select_disjoint_local_stitches(
    LOCAL_CANDIDATES, max_score=float(ASSOCIATION.get('max_assignment_score', 0.78))
)
LOCAL_COMPONENTS = build_local_components(TRACKLETS['tracklet_id'], LOCAL_MATCHES)
CROSS_TRACKLETS, CROSS_SAMPLES, CROSS_FLOOR_POINTS, CROSS_PIXEL_POINTS = aggregate_local_components(
    TRACKLETS, LOCAL_COMPONENTS, EMBEDDINGS, EMBEDDING_SAMPLES, TRACKLET_FLOOR_POINTS, TRACKLET_PIXEL_POINTS
)
CROSS_CANDIDATES = build_candidates(
    CROSS_TRACKLETS, CROSS_FLOOR_POINTS, CROSS_PIXEL_POINTS, CROSS_SAMPLES,
    include_same_camera=False, include_cross_camera=True,
)
CROSS_MATCHES, AMBIGUOUS_REJECTIONS = select_mutual_best_matches(
    CROSS_CANDIDATES, max_score=float(ASSOCIATION.get('max_assignment_score', 0.78)),
    min_margin=float(ASSOCIATION.get('min_match_margin', 0.05)),
)
CANDIDATES = pd.concat([LOCAL_CANDIDATES, CROSS_CANDIDATES], ignore_index=True, sort=False)
MATCHES = pd.concat([LOCAL_MATCHES, CROSS_MATCHES], ignore_index=True, sort=False).reset_index(drop=True)
REID_MAPPING, ACCEPTED_MATCHES, CLUSTER_REJECTIONS = cluster_matches(TRACKLETS, MATCHES)
REJECTED_MATCHES = pd.concat([AMBIGUOUS_REJECTIONS, CLUSTER_REJECTIONS], ignore_index=True, sort=False)
REID_MAPPING = annotate_mapping_quality(REID_MAPPING, ACCEPTED_MATCHES, REJECTED_MATCHES)
GLOBAL_TRACKS = BASE_EVENTS.merge(
    REID_MAPPING[['store_id', 'camera_id', 'local_track_id', 'global_track_id', 'is_cross_camera_identity']],
    on=['store_id', 'camera_id', 'local_track_id'], how='left', validate='many_to_one',
).sort_values(['camera_id', 'frame_index', 'local_track_id']).reset_index(drop=True)

GLOBAL_TRACKS.to_csv(TABLES_DIR / 'global_tracks.csv', index=False)
REID_MAPPING.to_csv(TABLES_DIR / 'reid_mapping.csv', index=False)
CANDIDATES.to_csv(TABLES_DIR / 'reid_candidates.csv', index=False)
ACCEPTED_MATCHES.to_csv(TABLES_DIR / 'reid_matches.csv', index=False)
REJECTED_MATCHES.to_csv(TABLES_DIR / 'reid_rejections.csv', index=False)
REID_REPORT = {
    'schema_version': 2,
    'status': 'association_complete',
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'tracking_run_id': TRACKING_RUN_ID,
    'identity_scope': 'estimated_global_reid_appearance_time' if IDENTITY_MODE == 'appearance_time' else 'global_reid',
    'identity_note': 'Cross-camera IDs are appearance-and-time estimates; analytics remains grouped by camera_track_uid.' if IDENTITY_MODE == 'appearance_time' else 'Cross-camera IDs use calibrated geometry plus appearance.',
    'identity_mode': IDENTITY_MODE,
    'embedding_cache': 'disabled',
    'camera_ids': sorted(GLOBAL_TRACKS['camera_id'].unique().tolist()),
    'cameras': int(GLOBAL_TRACKS['camera_id'].nunique()),
    'total_tracklets': int(len(TRACKLETS)),
    'embedded_tracklets': int(EMBEDDING_REPORT['embedded_tracklets']),
    'global_identities': int(REID_MAPPING['global_track_id'].nunique()),
    'cross_camera_identities': int(REID_MAPPING.loc[REID_MAPPING['is_cross_camera_identity'], 'global_track_id'].nunique()),
    'candidates': int(len(CANDIDATES)),
    'accepted_matches': int(len(ACCEPTED_MATCHES)),
    'rejected_matches': int(len(REJECTED_MATCHES)),
    'local_stitches': int((~ACCEPTED_MATCHES['is_cross_camera']).sum()) if not ACCEPTED_MATCHES.empty else 0,
    'ambiguous_rejections': int(REJECTED_MATCHES['rejection_reason'].isin(['ambiguous_margin', 'not_mutual_best']).sum()) if not REJECTED_MATCHES.empty else 0,
    'embedding_report': EMBEDDING_REPORT,
    'balanced_association': {
        'local_stitch_first': True,
        'same_camera_max_time_gap_sec': float(ASSOCIATION.get('same_camera_max_time_gap_sec', 10.0)),
        'same_camera_max_pixel_distance': float(ASSOCIATION.get('same_camera_max_pixel_distance', 180.0)),
        'synchronization_tolerance_sec': float(ASSOCIATION.get('synchronization_tolerance_sec', 0.35)),
        'min_synchronized_samples': int(ASSOCIATION.get('min_synchronized_samples', 3)),
        'min_match_margin': float(ASSOCIATION.get('min_match_margin', 0.05)),
        'allowed_camera_pairs': ASSOCIATION.get('allowed_camera_pairs', []),
    },
    'camera_time_offsets_sec': REID_CONFIG.get('synchronization', {}).get('camera_time_offsets_sec', {}),
    'timebase': TIMEBASE_REPORT,
}
write_json(TABLES_DIR / 'reid_report.json', REID_REPORT)
print('Estimated global identities: {:,}; cross-camera estimates: {:,}'.format(REID_REPORT['global_identities'], REID_REPORT['cross_camera_identities']))
display(REID_MAPPING.head())


RuntimeError: ReID stopped: correct the cameras marked false in reid_calibration_diagnostics.csv, then rerun this cell.

## Final outputs and dashboard videos

The dashboard uses global annotated videos only when this cell finishes every camera and writes a matching reid_run_manifest.json. If rendering fails, the app safely falls back to the normal local-tracking videos.


In [ ]:
from uuid import uuid4

zone_columns_with_global = [
    'tracking_run_id', 'store_id', 'camera_id', 'camera_track_uid', 'global_track_id', 'is_cross_camera_identity',
    'local_track_id', 'is_employee', 'is_customer', 'confidence', 'frame_index', 'timestamp_sec',
    'foot_x', 'foot_y', 'x1', 'y1', 'x2', 'y2', 'floor_x', 'floor_y', 'calibration_mode',
    'zone_id', 'zone_label_ar', 'zone_kind',
]
ZONE_EVENTS = GLOBAL_TRACKS[zone_columns_with_global].sort_values(['store_id', 'camera_id', 'camera_track_uid', 'frame_index']).reset_index(drop=True)
ZONE_EVENTS.to_csv(TABLES_DIR / 'zone_events.csv', index=False)

def write_zone_run_manifest(reid_display_ready):
    write_json(TABLES_DIR / 'camera_zone_run.json', {
        'schema_version': 2,
        'tracking_run_id': TRACKING_RUN_ID,
        'identity_scope': ('estimated_global_reid_display' if IDENTITY_MODE == 'appearance_time' else 'global_reid_display') if reid_display_ready else 'camera_local',
        'identity_note': 'global_track_id is estimated appearance-and-time display evidence only; retail analytics remains grouped by camera_track_uid' if IDENTITY_MODE == 'appearance_time' else 'global_track_id is display evidence only; retail analytics remains grouped by camera_track_uid',
        'camera_ids': sorted(ZONE_EVENTS['camera_id'].unique().tolist()),
        'zone_event_rows': int(len(ZONE_EVENTS)),
        'global_identities': int(ZONE_EVENTS['global_track_id'].nunique()),
    })

def artifact_entry(path):
    resolved = Path(path).resolve()
    return {
        'path': resolved.relative_to(PROJECT_ROOT).as_posix(),
        'size_bytes': int(resolved.stat().st_size),
        'sha256': sha256_file(resolved),
    }


def manifest_payload(status, video_outputs):
    artifact_paths = [TABLES_DIR / 'global_tracks.csv', TABLES_DIR / 'reid_mapping.csv', TABLES_DIR / 'reid_report.json']
    if status == 'complete':
        artifact_paths.extend(PROJECT_ROOT / row['path'] for row in video_outputs)
    artifacts = [artifact_entry(path) for path in artifact_paths if path.exists()]
    return {
        'schema_version': 3,
        'status': status,
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'tracking_run_id': TRACKING_RUN_ID,
        'identity_mode': IDENTITY_MODE,
        'identity_scope': 'estimated_global_reid_appearance_time' if IDENTITY_MODE == 'appearance_time' else 'global_reid',
        'camera_ids': sorted(GLOBAL_TRACKS['camera_id'].unique().tolist()),
        'global_tracks_path': 'Output/tables/global_tracks.csv',
        'mapping_path': 'Output/tables/reid_mapping.csv',
        'report_path': 'Output/tables/reid_report.json',
        'video_outputs': video_outputs,
        'artifacts': artifacts,
    }


def render_global_videos():
    source_videos = resolve_input_videos(GLOBAL_TRACKS['camera_id'].unique())
    staging_dir = VIDEOS_DIR / f'.reid_stage_{TRACKING_RUN_ID}_{uuid4().hex[:8]}'
    staging_dir.mkdir(parents=True, exist_ok=False)
    staged = []
    for camera_id, camera_rows in GLOBAL_TRACKS.groupby('camera_id', sort=True):
        display_rows = interpolate_display_rows(camera_rows, max_frame_gap=2)
        by_frame = {}
        for row in display_rows.to_dict('records'):
            by_frame.setdefault(int(row['frame_index']), []).append(row)
        capture = cv2.VideoCapture(str(source_videos[str(camera_id)]))
        if not capture.isOpened():
            raise RuntimeError(f'Could not open source video for rendering: {camera_id}')
        writer = None
        stage_path = staging_dir / f'global_annotated_{camera_id}.mp4'
        try:
            frame_index = 0
            while True:
                ok, frame = capture.read()
                if not ok or frame is None:
                    break
                if writer is None:
                    fps = float(capture.get(cv2.CAP_PROP_FPS))
                    if not np.isfinite(fps) or fps <= 0:
                        fps = float(VIDEO_METADATA.loc[VIDEO_METADATA['camera_id'] == camera_id, 'fps'].iloc[0])
                    writer = cv2.VideoWriter(str(stage_path), cv2.VideoWriter_fourcc(*'mp4v'), fps, (frame.shape[1], frame.shape[0]))
                    if not writer.isOpened():
                        raise RuntimeError(f'Could not create global annotated video for {camera_id}.')
                for row in by_frame.get(frame_index, []):
                    x1, y1, x2, y2 = map(int, map(round, (row['x1'], row['y1'], row['x2'], row['y2'])))
                    label_value = str(row['global_track_id']).replace('global_', '')
                    color = color_for_identity(str(row['global_track_id']))
                    label = f"{'EID' if IDENTITY_MODE == 'appearance_time' else 'GID'} {label_value}"
                    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                    cv2.putText(frame, label, (x1, max(18, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2, cv2.LINE_AA)
                writer.write(frame)
                frame_index += 1
        finally:
            capture.release()
            if writer is not None:
                writer.release()
        interpolated_rows = int(display_rows['display_interpolated'].sum())
        staged.append((camera_id, stage_path, VIDEOS_DIR / f'global_annotated_{camera_id}.mp4', interpolated_rows))
    return staged


if not as_bool(REID_CONFIG.get('execution', {}).get('render_global_videos', True)):
    write_json(REID_MANIFEST_PATH, manifest_payload('tables_complete', []))
    write_zone_run_manifest(False)
    print('Global tables and zone events are complete. Video rendering is disabled in mtmc_reid_config.json.')
else:
    write_json(REID_MANIFEST_PATH, manifest_payload('rendering', []))
    staged_videos = render_global_videos()
    video_outputs = []
    for camera_id, stage_path, final_path, interpolated_rows in staged_videos:
        stage_path.replace(final_path)
        video_outputs.append({'camera_id': camera_id, 'path': final_path.relative_to(PROJECT_ROOT).as_posix(), 'interpolated_display_rows': interpolated_rows})
    write_json(REID_MANIFEST_PATH, manifest_payload('complete', video_outputs))
    write_zone_run_manifest(True)
    print(f'Estimated ReID completed for {len(video_outputs)} cameras. The dashboard can now use the annotated videos.')
